### Fundus image measurements by AutoMorph

In [ ]:
%load_ext rpy2.ipython

%load_ext autoreload
%autoreload 2

### Import libraries from R and python
This notebook includes statistical modeling using R package mcgv.

In [ ]:
%%R
library('ggplot2')
library('nlme')
library("mgcv")
library("itsadug")
library("ggthemes")
theme_set(theme_minimal())

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import seaborn as sns
import statsmodels.api as sm

from utils import helpers, nako, plot

pd.options.display.float_format = "{:,.2f}".format
plot.set_rc_params(kind='paper', notebook_dpi=120)

In [ ]:
nako_paths = nako.get_nako_paths(dataset='590')
df_labels = pd.read_csv(nako_paths['labels_file'], index_col='ID')
decoder = nako.FeatureDecoder(nako_paths['metadata_file'])

results_file = 'data/rt_leftcentral.csv'
df = pd.read_csv(results_file)
df['Name'] = df['Name'].apply(lambda x: int(os.path.basename(x).split(sep='.')[0]))
df.rename(columns={'Name': 'ID'}, inplace=True)
df.set_index(['ID'], inplace=True)

feature_names = ['age', 'sex', 'smoking', 'hypertension', 'diabetes', 'systolic']
for feature, feature_code in zip(feature_names, ['basis_age', 'basis_sex', 'a_smok_stat_qn', 'a_rr_hypertens', 'd_an_met_1', 'a_rr_sys']):
    df[feature] = df_labels.loc[df.index][feature_code]
    mapping, _= decoder.decode(feature_code)
    df, mapping = decoder.group_nans(feature, mapping, df)
    df, mapping = decoder.modify_categorical(feature, mapping, df)
    if feature not in ['age', 'systolic']:
        mapping.pop(np.nan, None)
        df[feature] = df[feature].map(mapping)

measure_names = ['CDR_vertical', 'CRAE_Knudtson_zone_b', 'CRVE_Knudtson_zone_b', 'AVR_Knudtson_zone_b', 'Fractal_dimension_zone_b', 'Vessel_density_zone_b', 'Distance_tortuosity_zone_b']
measure_labels = ['Cup-to-disk ratio', 'Central Retinal Artery Equivalent [μm]', 'Central Retinal Vein Equivalent [μm]', 'Arteriolar-to-venular ratio', 'Fractal dimension', 'Vessel density', 'Distance tortuosity']
mapping_measures = dict((zip(measure_names, measure_labels)))
# df.rename(columns=mapping_measures, inplace=True)

### Check for missing values

In [ ]:
df_nans = df.isna().sum() / df.shape[0]

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
sns.barplot(x=df_nans.values, y=df_nans.index, ax=ax)
ax.set_xlabel('Percentage of missing values')
ax.set_ylabel('Feature')
plt.tight_layout()

### Check distribution

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(12, 5))
ax = ax.flatten()
for i, measure in enumerate(measure_names):
    df_subset = df.dropna(subset=[measure])
    df_out = helpers.remove_outliers_iqr(df_subset, measure)
    sns.kdeplot(x=measure, hue='sex', data=df_out, ax=ax[i])
    # sns.histplot(x=measure, data=df, ax=ax[i])
    sns.move_legend(ax[i], loc='upper right')
    ax[i].set_title(mapping_measures[measure])
    ax[i].set_xlabel('')

plt.tight_layout()

In [ ]:
for i, measure in enumerate(measure_names):
    fig, ax = plt.subplots(1, 1)
    df_subset = df.dropna(subset=[measure])
    df_out  = helpers.remove_outliers_iqr(df_subset, measure)
    # sns.kdeplot(x=measure, data=df_out, ax=ax)
    sns.histplot(x=measure, data=df_out, kde=True, stat='percent', bins=20, ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel('Percent', fontsize=12)
    ax.tick_params(labelsize=11)
    plt.tight_layout()
    fig.savefig(f'plots/measurements/{measure_names[i]}_dist.svg')

In [ ]:
# pd.options.display.float_format = "{:,.3f}".format
df['age class'] = pd.cut(df['age'], [19, 40, 60, 80])

for measure in measure_names:
    for feature in ['age class', 'sex', 'smoking', 'hypertension']:
        df_subset = df.dropna(subset=[measure, feature])[[measure, feature]]
        df_out = helpers.remove_outliers_iqr(df_subset, measure)

        df_groupby = df_out.groupby(feature)

        print(df_groupby.agg({measure: ['mean','std']}).transpose())
        print()
        break
    break

### QQ-plots

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(10, 5))
ax = ax.flatten()
for i, measure in enumerate(measure_names):
    df_subset = df.dropna(subset=[measure])
    df_out = helpers.remove_outliers_iqr(df_subset, measure)
    qq_plot = sm.qqplot(df_out[measure].to_numpy(), line ='45', fit=True, markersize=1,  ax=ax[i])
    ax[i].set_title(mapping_measures[measure])

plt.tight_layout()

### Select a measure to analyze

In [ ]:
measure = measure_names[5]
measure_label = mapping_measures[measure]
feature = feature_names[1]
print(measure, measure_label, feature, sep='\n')

### Remove outliers

In [ ]:
print('Percentage of samples after removing nans and outliers with iqr:')
df_subset = df.dropna(subset=[measure, feature])[feature_names + [measure]]
df_out = helpers.remove_outliers_iqr(df_subset, measure)
print(f'{df_subset.shape[0] * 100 / df.shape[0]:.2f}% > {df_out.shape[0] * 100 / df.shape[0]:.2f}%')

In [ ]:
# Ensure all columns are the same datatype
df_out = df_out.astype({measure: float, 'age': int})

with (ro.default_converter + pandas2ri.converter).context():
  r_df = ro.conversion.get_conversion().py2rpy(df_out)

%Rpush r_df measure measure_label feature

### Fit Generalized Additive Model (GAM) with a smooth term and a factor

In [ ]:
%%R
# Set sex as a factor and map to categorical values
r_df[,feature] = factor(r_df[,feature])

gam_formula_str = paste(measure, " ~ ", feature, " + s(age, by=", feature, ")", sep="")
gam_formula_label = paste(measure_label, " ~ ", feature, " + s(age, by=", feature, ")", sep="")
gam_formula = as.formula(gam_formula_str)
gam_model = gam(gam_formula, data=r_df, family=scat(link="identity"))
# summary(gam_model)

In [ ]:
%%R
summary(gam_model)

In [ ]:
%%R
gamtabs(gam_model)

In [ ]:
%%R
gam.check(gam_model)

#### Save plot values on a df to plot with python

In [ ]:
%%R
age_grid = seq(min(r_df$age), max(r_df$age))
# newdata = data.frame(age=age_grid, sex=levels(r_df$sex))
newdata = expand.grid(age=age_grid, sex = levels(r_df$sex))

# Predict with standard errors
pred = predict(gam_model, newdata, type="response", se.fit=TRUE)


z <- qnorm(0.975)

plot_df <- data.frame(
  age = newdata$age,
  sex = newdata$sex,
  fit = gam_model$family$linkinv(pred$fit),
  lower = gam_model$family$linkinv(pred$fit - z * pred$se.fit),
  upper = gam_model$family$linkinv(pred$fit + z * pred$se.fit)
)

filename = paste("./plots/measurements/", measure, "_", feature, ".csv", sep="")
write.csv(plot_df, filename, row.names = FALSE)

head(plot_df)

In [ ]:
%%R
# plot(gam_model)
svg(filename=paste("./plots/measurements/", measure, "_", feature, ".svg", sep=""), width=6.27, height=5.01, pointsize=16)


par(
    family = "Arial",
    las = 1,                 # axis labels horizontal
    cex = 1,                 # base font size (scaled via device pointsize)
    cex.axis = 1,            # tick labels
    cex.lab = 1,             # axis labels
    cex.main = 1.1,          # title size (11 vs 10)
    lwd = 0.5,               # line width
    tcl = -0.2,              # tick length (negative = inward)
    mar = c(4, 4, 1, 1),     # margins bottom, left, top, right
    mgp = c(2.5, 0.5, 0)       # label, tick label, line spacing
)

plot_smooth(gam_model, view='age', plot_all=feature, print.summary=FALSE, rug=FALSE, ylab=measure_label, xlab='', 
            col=c('#ff7f0e', '#1f77b4', '#2ca02c'), hide.label=TRUE, legend_plot_all='bottomright', yaxt='n')

# Reduce the number of y ticks by half
default_ticks <- axTicks(2)
reduced_ticks <- default_ticks[seq(1, length(default_ticks), by = 2)]
axis(2, at = reduced_ticks)

mtext('Age', side=1, line=1.8, cex=1)
dev.off()

### Fit Generalized Additive Model (GAM) for 2 continuous variables

In [ ]:
%%R
# Set sex as a factor and map to categorical values
r_df[,feature] = factor(r_df[,feature])

gam_formula_str = paste(measure, " ~ sex + te(age, systolic, by=sex)")
gam_formula_label = paste(measure_label, " ~ sex + te(age, systolic, by=sex)")
gam_formula = as.formula(gam_formula_str)
gam_model = gam(gam_formula, data=r_df, family=scat(link="identity"))
summary(gam_model)

In [ ]:
%%R
gamtabs(gam_model)

In [ ]:
%%R
gam.check(gam_model)

In [ ]:
%%R
png(filename = "./plots/temp_plot.png", width=800, height=400)
par(mfrow=c(1, 2), pty="s")
fvisgam(gam_model, view=c("age", "systolic"), cond=list(sex="Female"), main="Female", color="topo", dec=2)
fvisgam(gam_model, view=c("age", "systolic"), cond=list(sex="Male"), main="Male", color="topo", dec=2)
mtext(gam_formula_label, outer=TRUE,  cex=1.2, line=-1)
dev.off()

In [ ]:
%%R
png(filename = "./plots/temp_plot.png", width=400, height=400)
par(mar=c(5,4,4,4))
plot_diff2(gam_model, view=c("age", "systolic"), comp=list(sex=c("Female", "Male")), dec=2)
dev.off()

### Comparison of some features

In [ ]:
feature_names = ['sex', 'smoking', 'hypertension']

measure_names = ['CDR_vertical', 'CRAE_Knudtson_zone_b', 'AVR_Knudtson_zone_b', 'Vessel_density_zone_b']
measure_labels = ['Cup-to-disk ratio', 'Central Retinal Artery Equivalent [μm]', 'Arteriolar-to-venular ratio','Vessel density (1e-3)']

In [ ]:
fig, ax = plt.subplots(2, 2)
plot.set_figsize(fig, 'full', height_ratio=0.8)
ax = np.array(ax).flatten()

for i, measure_name in enumerate(measure_names):
    df_plot = pd.read_csv(os.path.join('plots', 'measurements', f'{measure_name}_{feature_names[0]}.csv'))
    if 'vessel' in measure_name.lower():
        df_plot['fit'] = 1e3 * df_plot['fit']
        df_plot['lower'] = 1e3 * df_plot['lower']
        df_plot['upper'] = 1e3 * df_plot['upper']

    sns.lineplot(data=df_plot, x='age', y='fit', hue='sex', linewidth=0.5, ax=ax[i])

    # Add confidence intervals
    for sex, g in df_plot.groupby('sex'):
        ax[i].fill_between(g['age'], g['lower'], g['upper'], alpha=0.25)

    ax[i].set_xlabel('Age')
    ax[i].set_ylabel(measure_labels[i])
    # ax[i].legend(loc='upper right')
    ax[i].get_legend().remove()
    plot.dettach_axes(ax[i])

ax[1].legend(loc='upper right')
# ax[3].legend(loc='lower left')

plot.set_labs(ax, panel_nums='auto', panel_num_pad=10)

plot.tight_layout()
fig.savefig('./plots/measurements.svg')
fig.savefig('plots/measurements.png', bbox_inches='tight')
